In [1]:
import json

In [2]:
# leemos el json 
archivo = open(file='instancia.json', mode='r', encoding= 'utf-8')
datos = json.load(archivo)

In [3]:
datos

{'parametros': {'num_maquinas': 2},
 'tareas': [{'id': 'T1', 'duracion': 4},
  {'id': 'T2', 'duracion': 2},
  {'id': 'T3', 'duracion': 7},
  {'id': 'T4', 'duracion': 5},
  {'id': 'T5', 'duracion': 1},
  {'id': 'T6', 'duracion': 3}]}

In [7]:
n = datos['parametros']['num_maquinas'] # int con n
tareas = datos['tareas'] # lista de diccionarios con la data de las tareas como id, duracion

tareas.sort(key= lambda x: x['duracion'], reverse= True) # ordenamos las tareas de mayor duracion a menor duracion

maquinas = [ {'id':i, "t_fin":0 , "historia": [] } for i in range(n)] #lista de maquinas donde tendremos registro de la data 

for tarea in tareas:
    # buscamos máquina disponible temprano
    maquina_menor_t = min(maquinas, key= lambda maquina:maquina['t_fin']) 
    
    t_inicio = maquina_menor_t['t_fin']
    t_fin = t_inicio + tarea['duracion']
    
    maquina_menor_t['historia'].append( 
                                    {'id': tarea['id'],
                                    't_inicio': t_inicio,
                                    't_fin': t_fin})
    maquina_menor_t['t_fin'] = t_fin
    
makespan = max(m['t_fin'] for m in maquinas)
makespan

for maquina in maquinas:
    print(f'Maquina: {maquina['id']} termina en el tiempo {maquina['t_fin']}')
    for tarea in maquina['historia']:
        print(f'->  La tarea {tarea['id']} comenzó en {tarea['t_inicio']} y terminó en {tarea['t_fin']}')
    

Maquina: 0 termina en el tiempo 11
->  La tarea T3 comenzó en 0 y terminó en 7
->  La tarea T6 comenzó en 7 y terminó en 10
->  La tarea T5 comenzó en 10 y terminó en 11
Maquina: 1 termina en el tiempo 11
->  La tarea T4 comenzó en 0 y terminó en 5
->  La tarea T1 comenzó en 5 y terminó en 9
->  La tarea T2 comenzó en 9 y terminó en 11


In [23]:
# PASO 1: Parsear el JSON
num_maquinas = datos['parametros']['num_maquinas']
tareas = datos['tareas']

# PASO 2: Ordenar las tareas (El corazón de la heurística LPT)
# Usamos sort() con una función lambda para decirle que ordene usando la 'duracion' de forma descendente (reverse=True)
tareas.sort(key=lambda x: x['duracion'], reverse=True)

# PASO 3: Inicializar el estado de las máquinas
# Creamos una lista de diccionarios. Cada máquina lleva el registro de en qué minuto se desocupa ('tiempo_fin')
maquinas = [{'id_maquina': i, 'tiempo_fin': 0, 'historial_tareas': []} for i in range(num_maquinas)]

# PASO 4: Asignar iterativamente
for tarea in tareas:
    # Magia de Python: min() busca la máquina que tenga el 'tiempo_fin' más pequeño
    maquina_libre = min(maquinas, key=lambda m: m['tiempo_fin'])
    
    # Calculamos en qué minuto empieza y termina esta tarea
    tiempo_inicio = maquina_libre['tiempo_fin']
    tiempo_fin = tiempo_inicio + tarea['duracion']
    
    # Guardamos el registro para el reporte final
    maquina_libre['historial_tareas'].append({
        'tarea': tarea['id'],
        'inicio': tiempo_inicio,
        'fin': tiempo_fin
    })
    
    # Actualizamos el reloj de la máquina para que sepa a qué hora se desocupa ahora
    maquina_libre['tiempo_fin'] = tiempo_fin

# PASO 5: Resultados y Makespan
# El Makespan es simplemente el tiempo de fin de la máquina que terminó al último
makespan = max(m['tiempo_fin'] for m in maquinas)

print(f"--- RESULTADOS DEL SCHEDULING ---")
print(f"Tiempo total de la operación (Makespan): {makespan} unidades de tiempo\n")

for m in maquinas:
    print(f"Máquina {m['id_maquina']} (Termina en el min {m['tiempo_fin']}):")
    for t in m['historial_tareas']:
        print(f"  -> {t['tarea']} (Inicio: {t['inicio']} | Fin: {t['fin']})")

--- RESULTADOS DEL SCHEDULING ---
Tiempo total de la operación (Makespan): 11 unidades de tiempo

Máquina 0 (Termina en el min 11):
  -> T3 (Inicio: 0 | Fin: 7)
  -> T6 (Inicio: 7 | Fin: 10)
  -> T5 (Inicio: 10 | Fin: 11)
Máquina 1 (Termina en el min 11):
  -> T4 (Inicio: 0 | Fin: 5)
  -> T1 (Inicio: 5 | Fin: 9)
  -> T2 (Inicio: 9 | Fin: 11)
